# RepurposeApp Video Worker — Free Google Colab T4 GPU Runner

This notebook runs the asynchronous video repurposing worker on a free Google Colab T4 GPU.
It polls jobs from your RepurposeApp dashboard, runs Whisper with CUDA acceleration (~10x faster than CPU), cuts 9:16 vertical promo reels with FFmpeg, and uploads the finished reels back to Cloudflare R2.

--- 

### 🔑 Setup: Add Your Secrets in Google Colab (One-time setup)
On the left sidebar, click the **🔑 Key icon (Secrets)** and add these keys with **Notebook access** enabled:

| Secret Name | Value |
| :--- | :--- |
| `GITHUB_TOKEN` | Your GitHub Personal Access Token (required for private repo clone) |
| `NEXT_PUBLIC_SUPABASE_URL` | Your Supabase project URL |
| `SUPABASE_SERVICE_ROLE_KEY` | Your Supabase service role key |
| `R2_ACCOUNT_ID` | Your Cloudflare R2 account ID |
| `R2_ACCESS_KEY_ID` | Your Cloudflare R2 access key ID |
| `R2_SECRET_ACCESS_KEY` | Your Cloudflare R2 secret access key |
| `GEMINI_API_KEY` | (Optional) Your Google Gemini API key for AI hooks |

> **GPU Check:** Ensure hardware acceleration is enabled: `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# 1. Authenticate, Clone & Install all GPU Dependencies
import os
import getpass
from google.colab import userdata

try:
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None

if not gh_token:
    print("💡 Tip: Add 'GITHUB_TOKEN' to Colab Secrets (🔑) to avoid pasting it each time.")
    gh_token = getpass.getpass("Enter GitHub Personal Access Token (or press Enter if repo is public): ").strip()

clone_url = f"https://{gh_token}@github.com/andreigrigore464-lang/RepurposeApp.git" if gh_token else "https://github.com/andreigrigore464-lang/RepurposeApp.git"

# Clean previous clone if any, clone target branch, install dependencies
!rm -rf RepurposeApp
!git clone -b feature/video-reels {clone_url}
%cd RepurposeApp/worker
!pip install -r requirements.txt
!pip install --upgrade "yt-dlp[default]"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

# Configure CUDA 12 shared libraries for CTranslate2 / Whisper GPU
!python3 -c "import os, glob, nvidia.cublas.lib, nvidia.cudnn.lib; [os.symlink(f, f'/usr/lib/x86_64-linux-gnu/{os.path.basename(f)}') for d in [os.path.dirname(nvidia.cublas.lib.__file__), os.path.dirname(nvidia.cudnn.lib.__file__)] for f in glob.glob(f'{d}/*.so*') if not os.path.exists(f'/usr/lib/x86_64-linux-gnu/{os.path.basename(f)}')];" 2>/dev/null || true
!ldconfig 2>/dev/null || true

In [ ]:
# 2. Sync Colab Secrets into worker environment & .env.local
import os
from google.colab import userdata

keys_to_export = [
    "NEXT_PUBLIC_SUPABASE_URL",
    "SUPABASE_SERVICE_ROLE_KEY",
    "R2_ACCOUNT_ID",
    "R2_ACCESS_KEY_ID",
    "R2_SECRET_ACCESS_KEY",
    "R2_BUCKET_NAME",
    "GEMINI_API_KEY"
]

env_lines = []
missing = []

for key in keys_to_export:
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = str(val)
            env_lines.append(f"{key}={val}")
            print(f"✓ {key}: configured")
        elif key not in ["R2_BUCKET_NAME", "GEMINI_API_KEY"]:
            missing.append(key)
    except Exception:
        if key not in ["R2_BUCKET_NAME", "GEMINI_API_KEY"]:
            missing.append(key)

# Write directly to worker/.env.local so CLI processes have immediate access
worker_dir = "/content/RepurposeApp/worker"
os.makedirs(worker_dir, exist_ok=True)
env_path = os.path.join(worker_dir, ".env.local")
with open(env_path, "w", encoding="utf-8") as f:
    f.write("\n".join(env_lines) + "\n")
print(f"✓ Wrote environment configuration to {env_path}")

# Check for optional YouTube cookies in Colab Secrets
try:
    yt_cookies = userdata.get("YOUTUBE_COOKIES")
    if yt_cookies:
        with open("/content/cookies.txt", "w", encoding="utf-8") as f:
            f.write(yt_cookies.strip())
        print("✓ YOUTUBE_COOKIES: synced to /content/cookies.txt")
except Exception:
    pass

if missing:
    print(f"\n⚠️ Missing {len(missing)} required secret(s) in Colab Secrets tab (🔑): {missing}")
    print("Please add them in the left sidebar under the 🔑 icon and turn ON 'Notebook access'.")
else:
    print("\n🚀 All credentials synced successfully! Ready to run.")

In [ ]:
# 3. Run the video worker daemon (listens for jobs live from your dashboard)
!python main.py